In [1]:
import numpy as np
from math import exp, log, sqrt, erf

# ============================================================
# Homework 2: Up-and-Out Call Pricing
# ============================================================

# 題目已指定的參數
S0 = 10
r = 0.04
T = 1 / 12
sigma = 0.40

# 題目沒有指定 K 和 L，因此自行假設
# K = 10：因為 S0 = 10，所以設定為 at-the-money
# L = 12：符合題目 L > K，且 barrier 不會太近或太遠
K = 10
L = 12

# 題目沒有指定模擬次數與時間切割數，因此自行假設
N = 100000      # 模擬路徑數
M = 60          # 時間切割數，用來檢查是否碰到 barrier
seed = 42       # 固定隨機種子，讓結果可重現


# ============================================================
# Helper Functions
# ============================================================

def normal_cdf(x):
    """Standard normal CDF."""
    return 0.5 * (1 + erf(x / sqrt(2)))


def black_scholes_call(S0, K, r, sigma, T):
    """European call price under Black-Scholes model."""
    d1 = (log(S0 / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * sqrt(T))
    d2 = d1 - sigma * sqrt(T)

    price = S0 * normal_cdf(d1) - K * exp(-r * T) * normal_cdf(d2)
    return price


def truncated_call_price(S0, K, L, r, sigma, T):
    """Price of modified control variate: e^{-rT}(S_T-K)1_{K<S_T<L}."""
    mu = log(S0) + (r - 0.5 * sigma ** 2) * T
    vol = sigma * sqrt(T)

    # P(K < S_T < L)
    zK = (log(K) - mu) / vol
    zL = (log(L) - mu) / vol
    prob = normal_cdf(zL) - normal_cdf(zK)

    # E[S_T * 1_{K < S_T < L}]
    dK = (mu + vol ** 2 - log(K)) / vol
    dL = (mu + vol ** 2 - log(L)) / vol

    expected_ST_indicator = exp(mu + 0.5 * vol ** 2) * (
        normal_cdf(dK) - normal_cdf(dL)
    )

    price = exp(-r * T) * (expected_ST_indicator - K * prob)
    return price


def simulate_gbm_paths(S0, r, sigma, T, N, M, seed=42, Z=None):
    """Simulate stock price paths under risk-neutral GBM."""
    if Z is None:
        rng = np.random.default_rng(seed)
        Z = rng.standard_normal((N, M))

    dt = T / M

    S = np.zeros((Z.shape[0], M + 1))
    S[:, 0] = S0

    # 使用 Black-Scholes 模型的 exact discretization 模擬股價路徑
    for t in range(M):
        S[:, t + 1] = S[:, t] * np.exp(
            (r - 0.5 * sigma ** 2) * dt
            + sigma * np.sqrt(dt) * Z[:, t]
        )

    return S


def up_and_out_payoff(S, K, L):
    """Compute up-and-out call payoff."""
    ST = S[:, -1]

    # 若路徑中任一時間點股價 >= L，表示 knock out
    knocked_out = np.max(S, axis=1) >= L

    # 若 knock out，payoff = 0；否則 payoff = max(S_T - K, 0)
    payoff = np.where(knocked_out, 0, np.maximum(ST - K, 0))
    return payoff


def summarize_result(samples):
    """Compute price, standard error, and 95% confidence interval."""
    price = np.mean(samples)
    sample_std = np.std(samples, ddof=1)
    standard_error = sample_std / np.sqrt(len(samples))

    ci_lower = price - 1.96 * standard_error
    ci_upper = price + 1.96 * standard_error
    ci_width = ci_upper - ci_lower

    return price, standard_error, ci_lower, ci_upper, ci_width



In [2]:
# ============================================================
# 1(a) Standard Monte Carlo
# ============================================================

def part_a_standard_mc():
    """1(a) Standard Monte Carlo."""
    discount = exp(-r * T)

    S = simulate_gbm_paths(S0, r, sigma, T, N, M, seed=seed)
    payoff = up_and_out_payoff(S, K, L)
    discounted_payoff = discount * payoff

    price, se, lower, upper, width = summarize_result(discounted_payoff)

    print("1(a) Standard Monte Carlo")
    print(f"Price = {price:.6f}")
    print(f"Standard Error = {se:.6f}")
    print(f"95% Confidence Interval = [{lower:.6f}, {upper:.6f}]")
    print(f"CI Width = {width:.6f}")
    print()

    return price, se, lower, upper, width


result_a = part_a_standard_mc()
   

1(a) Standard Monte Carlo
Price = 0.278562
Standard Error = 0.001454
95% Confidence Interval = [0.275712, 0.281412]
CI Width = 0.005700



In [3]:
# ============================================================
# 1(b) Antithetic Variates
# ============================================================

def part_b_antithetic():
    """1(b) Monte Carlo with antithetic variates."""
    discount = exp(-r * T)

    # 使用 N/2 組 pair，使總模擬路徑數約為 N
    n_pairs = N // 2

    rng = np.random.default_rng(seed)
    Z = rng.standard_normal((n_pairs, M))

    # 一條路徑使用 Z，另一條對偶路徑使用 -Z
    S_plus = simulate_gbm_paths(S0, r, sigma, T, n_pairs, M, Z=Z)
    S_minus = simulate_gbm_paths(S0, r, sigma, T, n_pairs, M, Z=-Z)

    payoff_plus = up_and_out_payoff(S_plus, K, L)
    payoff_minus = up_and_out_payoff(S_minus, K, L)

    # 每一組對偶 payoff 取平均
    pair_payoff = 0.5 * (payoff_plus + payoff_minus)
    discounted_pair_payoff = discount * pair_payoff

    price, se, lower, upper, width = summarize_result(discounted_pair_payoff)

    print("1(b) Antithetic Variates")
    print(f"Price = {price:.6f}")
    print(f"Standard Error = {se:.6f}")
    print(f"95% Confidence Interval = [{lower:.6f}, {upper:.6f}]")
    print(f"CI Width = {width:.6f}")
    print()

    return price, se, lower, upper, width


result_b = part_b_antithetic()

1(b) Antithetic Variates
Price = 0.278136
Standard Error = 0.001156
95% Confidence Interval = [0.275870, 0.280402]
CI Width = 0.004532



In [4]:
# ============================================================
# 1(c) European Call Control Variate
# ============================================================

def part_c_european_call_cv():
    """1(c) Control variate using European call."""
    discount = exp(-r * T)

    S = simulate_gbm_paths(S0, r, sigma, T, N, M, seed=seed)
    ST = S[:, -1]

    # 目標變數 Y：up-and-out call 的折現 payoff
    barrier_payoff = up_and_out_payoff(S, K, L)
    Y = discount * barrier_payoff

    # 控制變數 X：European call 的折現 payoff
    euro_payoff = np.maximum(ST - K, 0)
    X = discount * euro_payoff

    # European call 的理論價格 E[X]
    EX = black_scholes_call(S0, K, r, sigma, T)

    # 依照上課講義使用 theta 作為修正係數
    # theta = Cov(Y, X) / Var(X)
    theta = np.cov(Y, X, ddof=1)[0, 1] / np.var(X, ddof=1)

    # Modified control variate estimator:
    # Y_theta = Y + theta * (E[X] - X)
    Y_theta = Y + theta * (EX - X)

    price, se, lower, upper, width = summarize_result(Y_theta)

    print("1(c) European Call Control Variate")
    print(f"E[X] = European Call Price = {EX:.6f}")
    print(f"Theta = {theta:.6f}")
    print(f"Price = {price:.6f}")
    print(f"Standard Error = {se:.6f}")
    print(f"95% Confidence Interval = [{lower:.6f}, {upper:.6f}]")
    print(f"CI Width = {width:.6f}")
    print()

    return price, se, lower, upper, width, theta, EX

result_c = part_c_european_call_cv()



1(c) European Call Control Variate
E[X] = European Call Price = 0.476467
Theta = 0.290934
Price = 0.278148
Standard Error = 0.001289
95% Confidence Interval = [0.275622, 0.280674]
CI Width = 0.005052



In [5]:
# ============================================================
# 1(d) Modified Control Variate
# ============================================================

def part_d_modified_cv():
    """1(d) Modified control variate given by the homework."""
    discount = exp(-r * T)

    S = simulate_gbm_paths(S0, r, sigma, T, N, M, seed=seed)
    ST = S[:, -1]

    # 目標變數 Y：up-and-out call 的折現 payoff
    barrier_payoff = up_and_out_payoff(S, K, L)
    Y = discount * barrier_payoff

    # 控制變數 X：題目給定的 h(S_T)
    # h(S_T) = S_T - K, if K < S_T < L; otherwise 0
    modified_payoff = np.where((ST > K) & (ST < L), ST - K, 0)
    X = discount * modified_payoff

    # 控制變數的理論期望 E[X]
    EX = truncated_call_price(S0, K, L, r, sigma, T)

    # 依照上課講義使用 theta 作為修正係數
    # theta = Cov(Y, X) / Var(X)
    theta = np.cov(Y, X, ddof=1)[0, 1] / np.var(X, ddof=1)

    # Modified control variate estimator:
    # Y_theta = Y + theta * (E[X] - X)
    Y_theta = Y + theta * (EX - X)

    price, se, lower, upper, width = summarize_result(Y_theta)

    print("1(d) Modified Control Variate")
    print(f"E[X] = Modified Control Variate Price = {EX:.6f}")
    print(f"Theta = {theta:.6f}")
    print(f"Price = {price:.6f}")
    print(f"Standard Error = {se:.6f}")
    print(f"95% Confidence Interval = [{lower:.6f}, {upper:.6f}]")
    print(f"CI Width = {width:.6f}")
    print()

    return price, se, lower, upper, width, theta, EX

result_d = part_d_modified_cv()

1(d) Modified Control Variate
E[X] = Modified Control Variate Price = 0.336008
Theta = 0.725029
Price = 0.277564
Standard Error = 0.000837
95% Confidence Interval = [0.275924, 0.279204]
CI Width = 0.003280



In [7]:
# ============================================================
# Part 1(e): Baseline Comparison + Parameter Adjustment
# ============================================================

def run_four_methods_given_params(
    S0_input, K_input, L_input,
    r_input, sigma_input, T_input,
    N_input, M_input, seed_input=42
):
    """
    Run four methods under one parameter setting.
    """
    discount = exp(-r_input * T_input)

    # ----------------------------
    # 1(a) Standard Monte Carlo
    # ----------------------------
    S = simulate_gbm_paths(
        S0_input, r_input, sigma_input,
        T_input, N_input, M_input,
        seed=seed_input
    )

    payoff = up_and_out_payoff(S, K_input, L_input)
    Y_standard = discount * payoff
    price_a, se_a, lower_a, upper_a, width_a = summarize_result(Y_standard)

    # ----------------------------
    # 1(b) Antithetic Variates
    # ----------------------------
    n_pairs = N_input // 2

    rng = np.random.default_rng(seed_input)
    Z = rng.standard_normal((n_pairs, M_input))

    S_plus = simulate_gbm_paths(
        S0_input, r_input, sigma_input,
        T_input, n_pairs, M_input,
        Z=Z
    )

    S_minus = simulate_gbm_paths(
        S0_input, r_input, sigma_input,
        T_input, n_pairs, M_input,
        Z=-Z
    )

    payoff_plus = up_and_out_payoff(S_plus, K_input, L_input)
    payoff_minus = up_and_out_payoff(S_minus, K_input, L_input)

    pair_payoff = 0.5 * (payoff_plus + payoff_minus)
    Y_antithetic = discount * pair_payoff
    price_b, se_b, lower_b, upper_b, width_b = summarize_result(Y_antithetic)

    # ----------------------------
    # 1(c) European Call Control Variate
    # ----------------------------
    ST = S[:, -1]

    Y = Y_standard
    X_euro = discount * np.maximum(ST - K_input, 0)
    EX_euro = black_scholes_call(S0_input, K_input, r_input, sigma_input, T_input)

    theta_euro = np.cov(Y, X_euro, ddof=1)[0, 1] / np.var(X_euro, ddof=1)
    Y_euro_cv = Y + theta_euro * (EX_euro - X_euro)

    price_c, se_c, lower_c, upper_c, width_c = summarize_result(Y_euro_cv)

    # ----------------------------
    # 1(d) Modified Control Variate
    # ----------------------------
    X_modified = discount * np.where(
        (ST > K_input) & (ST < L_input),
        ST - K_input,
        0
    )

    EX_modified = truncated_call_price(
        S0_input, K_input, L_input,
        r_input, sigma_input, T_input
    )

    theta_modified = np.cov(Y, X_modified, ddof=1)[0, 1] / np.var(X_modified, ddof=1)
    Y_modified_cv = Y + theta_modified * (EX_modified - X_modified)

    price_d, se_d, lower_d, upper_d, width_d = summarize_result(Y_modified_cv)

    results = [
        {
            "Method": "Standard MC",
            "Price": price_a,
            "SE": se_a,
            "CI Lower": lower_a,
            "CI Upper": upper_a,
            "CI Width": width_a,
            "Theta": None
        },
        {
            "Method": "Antithetic Variates",
            "Price": price_b,
            "SE": se_b,
            "CI Lower": lower_b,
            "CI Upper": upper_b,
            "CI Width": width_b,
            "Theta": None
        },
        {
            "Method": "European Call CV",
            "Price": price_c,
            "SE": se_c,
            "CI Lower": lower_c,
            "CI Upper": upper_c,
            "CI Width": width_c,
            "Theta": theta_euro
        },
        {
            "Method": "Modified CV",
            "Price": price_d,
            "SE": se_d,
            "CI Lower": lower_d,
            "CI Upper": upper_d,
            "CI Width": width_d,
            "Theta": theta_modified
        }
    ]

    return results

def baseline_comparison_table():
    """Print baseline comparison table for 1(a) to 1(d)."""

    print("=" * 120)
    print("Baseline Results for 1(a) to 1(d)")
    print("=" * 120)
    print(f"Parameter setting: S0={S0}, K={K}, L={L}, r={r}, sigma={sigma}, T={T}, N={N}, M={M}")
    print("=" * 120)
    print()

    result_a = part_a_standard_mc()
    result_b = part_b_antithetic()
    result_c = part_c_european_call_cv()
    result_d = part_d_modified_cv()

    baseline_results = [
        {
            "Method": "Standard MC",
            "Price": result_a[0],
            "SE": result_a[1],
            "CI Lower": result_a[2],
            "CI Upper": result_a[3],
            "CI Width": result_a[4],
            "Theta": None
        },
        {
            "Method": "Antithetic Variates",
            "Price": result_b[0],
            "SE": result_b[1],
            "CI Lower": result_b[2],
            "CI Upper": result_b[3],
            "CI Width": result_b[4],
            "Theta": None
        },
        {
            "Method": "European Call CV",
            "Price": result_c[0],
            "SE": result_c[1],
            "CI Lower": result_c[2],
            "CI Upper": result_c[3],
            "CI Width": result_c[4],
            "Theta": result_c[5]
        },
        {
            "Method": "Modified CV",
            "Price": result_d[0],
            "SE": result_d[1],
            "CI Lower": result_d[2],
            "CI Upper": result_d[3],
            "CI Width": result_d[4],
            "Theta": result_d[5]
        }
    ]

    print("=" * 120)
    print("Baseline Comparison Table for 1(a) to 1(d)")
    print("=" * 120)
    print(
        f"{'Method':25s} "
        f"{'Price':>12s} "
        f"{'SE':>12s} "
        f"{'CI Lower':>12s} "
        f"{'CI Upper':>12s} "
        f"{'CI Width':>12s} "
        f"{'Theta':>12s}"
    )
    print("-" * 120)

    for row in baseline_results:
        theta_text = "" if row["Theta"] is None else f"{row['Theta']:.6f}"

        print(
            f"{row['Method']:25s} "
            f"{row['Price']:12.6f} "
            f"{row['SE']:12.6f} "
            f"{row['CI Lower']:12.6f} "
            f"{row['CI Upper']:12.6f} "
            f"{row['CI Width']:12.6f} "
            f"{theta_text:>12s}"
        )

    print("=" * 120)
    print()

    return baseline_results

def print_result_table(title, fixed_description, adjusted_description, all_results):
    """
    Print a clean summary table.
    """
    print("=" * 120)
    print(title)
    print("=" * 120)
    print(fixed_description)
    print(adjusted_description)
    print("-" * 120)
    print(
        f"{'Adjusted':>10s} "
        f"{'Method':25s} "
        f"{'Price':>12s} "
        f"{'SE':>12s} "
        f"{'CI Lower':>12s} "
        f"{'CI Upper':>12s} "
        f"{'CI Width':>12s} "
        f"{'Theta':>12s}"
    )
    print("-" * 120)

    for row in all_results:
        theta_text = "" if row["Theta"] is None else f"{row['Theta']:.6f}"

        print(
            f"{str(row['Adjusted']):>10s} "
            f"{row['Method']:25s} "
            f"{row['Price']:12.6f} "
            f"{row['SE']:12.6f} "
            f"{row['CI Lower']:12.6f} "
            f"{row['CI Upper']:12.6f} "
            f"{row['CI Width']:12.6f} "
            f"{theta_text:>12s}"
        )

    print("=" * 120)
    print()


def part_e_parameter_adjustment():
    """
    1(e) Two-part parameter adjustment.

    Table 1:
        Fix N = 100000 and adjust L = 11, 12, 13.

    Table 2:
        Fix L = 12 and adjust N = 10000, 50000, 100000.
    """

    # 題目指定參數
    S0_base = 10
    r_base = 0.04
    T_base = 1 / 12
    sigma_base = 0.40

    # 自行假設並固定的參數
    K_base = 10
    M_base = 60
    seed_base = 42

    # ========================================================
    # Table 1: Adjust L, fix N
    # ========================================================

    fixed_N = 100000
    L_values = [11, 12, 13]

    results_L = []

    for L_now in L_values:
        results = run_four_methods_given_params(
            S0_input=S0_base,
            K_input=K_base,
            L_input=L_now,
            r_input=r_base,
            sigma_input=sigma_base,
            T_input=T_base,
            N_input=fixed_N,
            M_input=M_base,
            seed_input=seed_base
        )

        for res in results:
            results_L.append({
                "Adjusted": f"L={L_now}",
                **res
            })

    print_result_table(
        title="Table 1: Adjust Barrier Level L",
        fixed_description=(
            f"Fixed parameters: S0={S0_base}, K={K_base}, r={r_base}, "
            f"sigma={sigma_base}, T={T_base}, N={fixed_N}, M={M_base}"
        ),
        adjusted_description="Adjusted parameter: L = 11, 12, 13",
        all_results=results_L
    )

    # ========================================================
    # Table 2: Adjust N, fix L
    # ========================================================

    fixed_L = 12
    N_values = [10000, 50000, 100000]

    results_N = []

    for N_now in N_values:
        results = run_four_methods_given_params(
            S0_input=S0_base,
            K_input=K_base,
            L_input=fixed_L,
            r_input=r_base,
            sigma_input=sigma_base,
            T_input=T_base,
            N_input=N_now,
            M_input=M_base,
            seed_input=seed_base
        )

        for res in results:
            results_N.append({
                "Adjusted": f"N={N_now}",
                **res
            })

    print_result_table(
        title="Table 2: Adjust Number of Simulations N",
        fixed_description=(
            f"Fixed parameters: S0={S0_base}, K={K_base}, L={fixed_L}, "
            f"r={r_base}, sigma={sigma_base}, T={T_base}, M={M_base}"
        ),
        adjusted_description="Adjusted parameter: N = 10000, 50000, 100000",
        all_results=results_N
    )

    return results_L, results_N

# ============================================================
# Run Part 1(e)
# ============================================================

# 先輸出固定 baseline 下，1(a) 到 1(d) 的比較表
baseline_results = baseline_comparison_table()

# 再輸出第 1(e) 的參數調整表格
# Table 1: 固定 N，調整 L
# Table 2: 固定 L，調整 N
results_L, results_N = part_e_parameter_adjustment()

Baseline Results for 1(a) to 1(d)
Parameter setting: S0=10, K=10, L=12, r=0.04, sigma=0.4, T=0.08333333333333333, N=100000, M=60

1(a) Standard Monte Carlo
Price = 0.278562
Standard Error = 0.001454
95% Confidence Interval = [0.275712, 0.281412]
CI Width = 0.005700

1(b) Antithetic Variates
Price = 0.278136
Standard Error = 0.001156
95% Confidence Interval = [0.275870, 0.280402]
CI Width = 0.004532

1(c) European Call Control Variate
E[X] = European Call Price = 0.476467
Theta = 0.290934
Price = 0.278148
Standard Error = 0.001289
95% Confidence Interval = [0.275622, 0.280674]
CI Width = 0.005052

1(d) Modified Control Variate
E[X] = Modified Control Variate Price = 0.336008
Theta = 0.725029
Price = 0.277564
Standard Error = 0.000837
95% Confidence Interval = [0.275924, 0.279204]
CI Width = 0.003280

Baseline Comparison Table for 1(a) to 1(d)
Method                           Price           SE     CI Lower     CI Upper     CI Width        Theta
------------------------------------------